# UCS420: Cognitive Computing — Assignment 4
## A Cognitive FAQ System Using Pandas (Nova 2.0)

**Note:** Enter your roll number in the `ROLL_NUMBER` variable below before running the notebook.

In [ ]:
import pandas as pd

ROLL_NUMBER = "YOUR_ROLL_NUMBER"  # Replace with your college roll number
last_two = str(1024170008)[-2:]

categories = ["billing", "account", "general"]
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]

personalized = []
if last_two.isdigit():
    for d in map(int, last_two):
        category = categories[d % 3]
        if category == "billing":
            q, a, k = "how can i check my payment status", "You can check your payment status in the billing section.", "payment status billing transaction"
        elif category == "account":
            q, a, k = "how do i update my registered mobile number", "You can update your registered mobile number from account settings.", "mobile number update account"
        else:
            q, a, k = "how can i contact customer support", "You can contact customer support during working hours.", "support help contact"
        personalized.append({"question": q, "answer": a, "keywords": k, "category": category})
else:
    personalized = [
        {"question": "personalized entry 1 — enter a valid roll number", "answer": "Replace ROLL_NUMBER with your roll number and rerun this cell.", "keywords": "roll number setup", "category": "general"},
        {"question": "personalized entry 2 — enter a valid roll number", "answer": "Replace ROLL_NUMBER with your roll number and rerun this cell.", "keywords": "roll number setup", "category": "general"},
    ]

df = pd.DataFrame(fixed_entries + personalized)
print(df)


## Q2 — Generate and Score a Hypothesis

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        text_words = set((row['question'] + ' ' + row['keywords']).lower().split())
        score = len(query_words & text_words)
        if score > 0:
            results.append((score, row['question'], row['answer'], row['category']))
    results.sort(key=lambda x: x[0], reverse=True)
    return pd.DataFrame(results, columns=['confidence', 'question', 'answer', 'category'])

query = input("Enter your query: ")
print(score_query(query, df))


## Q3 — Find Questions in the Same Category

In [ ]:
def same_category(category_name, df):
    return df[df['category'].str.lower() == category_name.lower()][['question', 'answer', 'keywords', 'category']]

personalized_category = df.iloc[4]['category']
print("Category:", personalized_category)
print(same_category(personalized_category, df))


## Q4 — Add a Keyword and Save the Updated DataFrame

In [ ]:
entry_index = 0
new_keyword = input("Enter a new keyword for the first FAQ entry: ").strip()
if new_keyword:
    df.at[entry_index, 'keywords'] = df.at[entry_index, 'keywords'] + ' ' + new_keyword

csv_filename = f"{ROLL_NUMBER}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Updated DataFrame saved as {csv_filename}")
print(df)


## Q5 — Count FAQ Entries Per Category

In [ ]:
category_counts = df.groupby('category').size()
print(category_counts)


## Q6 — Handle Ties in the Highest Confidence Score

In [ ]:
def score_query_with_ties(query, df):
    results = score_query(query, df)
    if results.empty:
        print("No matching entries found.")
        return results

    highest = results['confidence'].max()
    top_matches = results[results['confidence'] == highest]
    print(f"Highest confidence: {highest}")
    if len(top_matches) > 1:
        print("Tie detected — all equally good matches:")
    else:
        print("Single best match:")
    print(top_matches)
    return top_matches

print("--- Tie demonstration ---")
score_query_with_ties("fee", df)

print("\n--- Non-tie demonstration ---")
score_query_with_ties("password reset login", df)


## Assignment Complete
All six questions are implemented in this single notebook because Q2–Q6 depend on the DataFrame created in Q1.